# Import Raw Data

Import the raw data from NearSpace Launch to be cleaned for future data analysis.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel(r"C:\Users\n_jac\RadStar\radStar.xlsx")
df_copy = df

In [ ]:
df

The raw data contains 8768 rows (548 packets) of data from January 2nd, 2025 to April 10th, 2025.

# Clean Data

## Change Data Types

All 10 of their radiation collection data columns are in float format despite the data being whole numbers, so I changed the column's types from float to integer. Then the "In Shadow" column is 0 if the satellite can see the sun and 1 if the satellite is in the shadow of the Earth, so I turned those values to integers for future analysis.

In [ ]:
df_copy["proton 0"] = df_copy["proton 0"].astype("Int64")
df_copy["proton 1"] = df_copy["proton 1"].astype("Int64")
df_copy["electron 0"] = df_copy["electron 0"].astype("Int64")
df_copy["electron 1"] = df_copy["electron 1"].astype("Int64")
df_copy["xray 0"] = df_copy["xray 0"].astype("Int64")
df_copy["xray 1"] = df_copy["xray 1"].astype("Int64")
df_copy["xray 2"] = df_copy["xray 2"].astype("Int64")
df_copy["xray 3"] = df_copy["xray 3"].astype("Int64")
df_copy["ses 0"] = df_copy["ses 0"].astype("Int64")
df_copy["ses 1"] = df_copy["ses 1"].astype("Int64")
df_copy["In Shadow"] = df_copy["In Shadow"].astype("Int64")
df_copy["timestamp"] = pd.to_datetime(df_copy["timestamp"]).dt.to_pydatetime()

## Rename Columns

We decided to rename the columns to make them easier to recall later on. Their origanal column names had capital letters and spaces which made the column harder to recall as we were making visuals.

In [ ]:
df_copy.columns = ["packetID", "sample", "uptime", "speedmode", "timestamp", "proton0", "proton1", "electron0", "electron1",
                    "xray0", "xray1", "xray2", "xray3", "ses0", "ses", "group", "lat", "lon", "alt", "in_shadow"]

## Finding Bad Packets

The next step of cleaning the data was to identify the packets that contained bad data that needed to be removed before moving forward as to not skew the results of our analysis. In the original data set, there appeared to be a few glitches in the data collection software. These glitches caused the timestamp to jump backwards, which could cause the whole packet's data to be duplicated a second time or for the data to overlap with another pack which caused the data collection to glitch out and become inaccurate.

In [ ]:
decreases = []

for i in range(1, len(df)):
    if df_copy.loc[i, 'timestamp'] < df_copy.loc[i-1, 'timestamp']:
        decreases.append(int(df_copy.loc[i, 'packetID']))

print(decreases)

These are the 23 packets that we identified for the timestamp jumping back in time that needed to be removed for the cleaned data set to be used for future analysis.

In [ ]:
for id in decreases:
    indexR = df_copy[df_copy['packetID'] == id].index[0]
    indexO = indexR - 16
    rowR = df_copy.loc[indexR].drop(['packetID','group','sample'])
    rowO = df_copy.loc[indexO].drop(['packetID','group','sample'])
    if rowO.equals(rowR):
        print("Packet " + str(df_copy.loc[indexO, 'packetID']) + " is identical to packet " + str(df_copy.loc[indexR, 'packetID']))
    else:
        print("Packet " + str(df_copy.loc[indexO, 'packetID']) + " overlaps with packet " + str(df_copy.loc[indexR, 'packetID']))

This identified if the packet was a full duplicate of the previous packet or if the packet overlapped with the previous packet causing a software glitch.

## Drop Bad Data

Now we drop the 23 bad packets that we identified. Then there were a few additional rows that contained null values for the column count categories, so we dropped those as well since we could not think of a clean way to fill them. Ultimately we dropped 416 rows of data, which is 4.74% of the raw data and comes in under the 5% threshold.

In [ ]:
df_filt = df_copy[~df_copy["packetID"].isin(decreases)]
df_filt = df_filt.dropna(subset=["electron0"])
df_filt = df_filt.drop(columns=["ses0"])

## Add New Columns

After dropping the bad packets of data, we needed to regroup the data. Groups contain all the observations that are within 5 minutes of each other, so if one observation to the next is greater than 5 minutes it will start a new group. Then we created a new ID column that was a string representing both the group number and observation number. We created a new column that was just the month in which the observation occured so that we could analyze trends by month later on. We created a total radiation column that summed the 'xray0', 'electron0', and 'proton0' values from each row. Finally, we standardized all of our radiation count columns to get a counts per second columns.

In [ ]:
divisors = {0: 4, 1: 16, 2: 64}

df_filt['group'] = ((df_filt['timestamp'].diff() > pd.Timedelta(minutes=5)).cumsum() + 1)
df_filt['obs_id'] = (df_filt['group'].astype(str).str.zfill(3) + (df_filt.groupby('group').cumcount() + 1).astype(str).str.zfill(3))
df_filt['month'] = df_filt["timestamp"].astype("str").str[5:7].astype("int")

df_filt["month"] = df_filt["month"].case_when(
    caselist=[
        (df_filt["month"] == 1, "Jan"),
        (df_filt["month"] == 2, "Feb"),
        (df_filt["month"] == 3, "Mar"),
        (df_filt["month"] == 4, "Apr")
    ]
)

df_filt["proton0_ps"] = df_filt["proton0"] / df_filt["speedmode"].map(divisors)
df_filt["proton1_ps"] = df_filt["proton1"] / df_filt["speedmode"].map(divisors)
df_filt["electron0_ps"] = df_filt["electron0"] / df_filt["speedmode"].map(divisors)
df_filt["electron1_ps"] = df_filt["electron1"] / df_filt["speedmode"].map(divisors)
df_filt["xray0_ps"] = df_filt["xray0"] / df_filt["speedmode"].map(divisors)
df_filt["xray1_ps"] = df_filt["xray1"] / df_filt["speedmode"].map(divisors)
df_filt["xray2_ps"] = df_filt["xray2"] / df_filt["speedmode"].map(divisors)
df_filt["xray3_ps"] = df_filt["xray3"] / df_filt["speedmode"].map(divisors)
df_filt["ses_ps"] = df_filt["ses"] / df_filt["speedmode"].map(divisors)

df_filt["total_radiation"] = df_filt["proton0"] + df_filt["electron0"] + df_filt["xray0"]
df_filt["total_radiation_ps"] = df_filt["total_radiation"] / df_filt["speedmode"].map(divisors)

In [ ]:
df_filt.info()

Our final cleaned data set contains 8352 rows of data and 30 columns.

# Export Clean Data

Export the cleaned data to be used in the other data analysis notebooks.

In [ ]:
df_filt.to_excel("cleaned_data.xlsx", index=False)

# Add kp Values

In [ ]:
kp = pd.read_excel("radstar_kp.xlsx")

In [ ]:
#Create a new, empty DataFrame with the columns we plan to have in our cleaned DataFrame
new_kp = pd.DataFrame(columns=["timestamp", "kp", "Ap"])

#Loop through all rows of the original kp DataFrame
for idx, row in kp.iterrows():
    #Loop through each row 8 times to grab each of the 8 kp values individually
    for i in range(8):
        #Create a temporary, 1-row DataFrame that holds a timestamp, kp value, and Ap value so that our new dataset has only one kp value per row
        temp = pd.DataFrame([{
                "timestamp": pd.to_datetime({"year": [2025],                #All of the data collection takes place in 2025
                                             "month": [kp.iloc[idx, 1]],    #Column 1 is the month
                                             "day": [kp.iloc[idx, 2]],      #Column 2 is the day
                                             "hour": [3+(i*3)]              #This gives us every 3 hours starting at 3 (kp is measured 3-hourly)
                                            })[0],
                "kp": kp.iloc[idx, 3+i],    #Grab the next kp value
                "Ap": kp.iloc[idx, -1]      #The last row is the Ap value. All 8 kp values will be stored with this value
        }])
        #Concatinate our temporary DataFrame with our new one
        new_kp = pd.concat([new_kp, temp], ignore_index=True)

In [ ]:
new_kp.to_excel("new_kp.xlsx", index=False)

#### HOW THE BELOW CODE WORKS:
- Both files are ordered chronologically (after cleaning clean_rad)
- Loop through the items in clean_rad until it encounters a kp value with a timestamp before the clean_rad row
    - If the clean_rad row happens before the kp timestamp, that means it's within that kp value's 3-hour block and will recieve that kp value
    - If the clean_rad row happens after the kp timestamp, we have made it past that kp index chronologically and can move onto the next one
- Once we made it to the end of the one of the DataFrames, we have assigned a kp value to all clean_rad rows

In [ ]:
kp_idx = 0          #The index for what kp value we are on
clean_idx = 0       #The index for which clean_rad row we are on

#Loop through the entire document until we're either out of kp values to use or out of rows to assign kp vlaues to
while (clean_idx < len(df_filt)) and (kp_idx < len(new_kp)):
    if df_filt.iloc[clean_idx, 4] < new_kp.iloc[kp_idx, 0]:                       #clean_rad column 4 is timestamp, while new_kp timestamp column 0 is timestamp
        df_filt.loc[df_filt.index[clean_idx], "kp"] = new_kp.iloc[kp_idx, 1]    #Assign the current kp value to the current clean_rad row
        df_filt.loc[df_filt.index[clean_idx], "Ap"] = new_kp.iloc[kp_idx, 2]    #Assign the current Ap value to the current clean_rad row
        clean_idx += 1                                                              #Move on to the next clean_rad row
    else:
        kp_idx += 1                                                                 #Move on to the next kp value

In [ ]:
df_filt.to_excel("clean_data_kp.xlsx", index=False)